In [1]:
# %% [markdown]
# # Multimodal Social-LightGCN for Yelp Recommendations
# This notebook implements a Graph Convolutional Network that fuses User-Item interactions 
# with the User-User social network (friends) from the Yelp Academic Dataset.

# %%
import os
import json
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
import time

# Configuration
class Args:
    user_json = "yelp_academic_dataset_user.json"
    review_json = "yelp_academic_dataset_review.json"
    business_json = "yelp_academic_dataset_business.json"
    max_reviews = 100000  # Adjust based on RAM
    epochs = 10
    batch_size = 2048
    lr = 0.001
    emb_dim = 64
    n_layers = 2
    device = "cuda" if torch.cuda.is_available() else "cpu"
    checkpoint_path = "social_lightgcn_model.pt"

args = Args()

# %% [markdown]
# ## 1. Data Loading & Preprocessing
# We parse the social network from the user file and interactions from the review file.

# %%
def load_yelp_data(args):
    print("Loading reviews...")
    # Loading reviews in chunks for memory efficiency
    review_chunks = pd.read_json(args.review_json, lines=True, chunksize=args.max_reviews)
    df_review = next(review_chunks)
    
    print("Loading users and social network...")
    # We only need user_id and friends from the user file
    user_map = {uid: i for i, uid in enumerate(df_review['user_id'].unique())}
    item_map = {bid: i for i, bid in enumerate(df_review['business_id'].unique())}
    
    # Extract edges
    u_idx = df_review['user_id'].map(user_map).values
    i_idx = df_review['business_id'].map(item_map).values
    
    # Load social network (friends)
    social_edges = []
    user_chunks = pd.read_json(args.user_json, lines=True, chunksize=50000)
    for chunk in user_chunks:
        # Filter for users we actually have in our review subset
        active_users = chunk[chunk['user_id'].isin(user_map.keys())]
        for _, row in active_users.iterrows():
            u_src = user_map[row['user_id']]
            friends = row['friends'].split(', ')
            for f in friends:
                if f in user_map:
                    social_edges.append([u_src, user_map[f]])
    
    social_edges = np.array(social_edges) if social_edges else np.empty((0, 2))
    return u_idx, i_idx, social_edges, user_map, item_map

u_idx, i_idx, social_edges, user_map, item_map = load_yelp_data(args)
num_users, num_items = len(user_map), len(item_map)
print(f"Users: {num_users}, Items: {num_items}, Social Edges: {len(social_edges)}")

# %% [markdown]
# ## 2. Graph Construction
# We normalize the adjacency matrices for both Interaction and Social graphs.

# %%
def normalize_adj(edge_index, shape):
    row, col = edge_index[0], edge_index[1]
    deg = torch.zeros(shape[0]).to(args.device)
    deg.index_add_(0, row, torch.ones(row.shape[0]).to(args.device))
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0
    norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
    return norm

# Prepare Tensors
edge_ui = torch.tensor(np.stack([u_idx, i_idx]), dtype=torch.long).to(args.device)
edge_uu = torch.tensor(social_edges.T, dtype=torch.long).to(args.device) if social_edges.size > 0 else None

norm_ui = normalize_adj(edge_ui, (num_users, num_items))
if edge_uu is not None:
    norm_uu = normalize_adj(edge_uu, (num_users, num_users))

# %% [markdown]
# ## 3. Social-LightGCN Model
# Extending LightGCN to propagate messages across the social graph.

# %%
class SocialLightGCN(nn.Module):
    def __init__(self, num_users, num_items, emb_dim, n_layers):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.n_layers = n_layers
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def computer(self, edge_ui, norm_ui, edge_uu=None, norm_uu=None):
        users_emb = self.user_emb.weight
        items_emb = self.item_emb.weight
        all_emb = [torch.cat([users_emb, items_emb], dim=0)]
        
        g_emb = all_emb[0]
        for _ in range(self.n_layers):
            # 1. Interaction Propagation (User <-> Item)
            u_idx, i_idx = edge_ui[0], edge_ui[1]
            # Items to Users
            new_u = torch.zeros_like(users_emb)
            new_u.index_add_(0, u_idx, norm_ui.unsqueeze(1) * items_emb[i_idx])
            
            # Users to Items
            new_i = torch.zeros_like(items_emb)
            new_i.index_add_(0, i_idx, norm_ui.unsqueeze(1) * users_emb[u_idx])
            
            # 2. Social Propagation (User <-> User)
            if edge_uu is not None:
                u_src, u_dst = edge_uu[0], edge_uu[1]
                social_u = torch.zeros_like(users_emb)
                social_u.index_add_(0, u_src, norm_uu.unsqueeze(1) * users_emb[u_dst])
                new_u = new_u + social_u # Fuse social influence
            
            users_emb, items_emb = new_u, new_i
            all_emb.append(torch.cat([users_emb, items_emb], dim=0))
            
        final_emb = torch.stack(all_emb, dim=1).mean(dim=1)
        return torch.split(final_emb, [self.num_users, self.num_items])

    def forward(self, u, i_pos, i_neg, edge_ui, norm_ui, edge_uu, norm_uu):
        sig_u, sig_i = self.computer(edge_ui, norm_ui, edge_uu, norm_uu)
        u_emb = sig_u[u]
        pos_emb = sig_i[i_pos]
        neg_emb = sig_i[i_neg]
        
        pos_scores = torch.mul(u_emb, pos_emb).sum(dim=1)
        neg_scores = torch.mul(u_emb, neg_emb).sum(dim=1)
        
        loss = -torch.mean(torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-10))
        # Regularization
        reg_loss = (1/2)*(u_emb.norm(2).pow(2) + pos_emb.norm(2).pow(2) + neg_emb.norm(2).pow(2)) / float(len(u))
        return loss + 1e-4 * reg_loss

# %% [markdown]
# ## 4. Training Loop

# %%
model = SocialLightGCN(num_users, num_items, args.emb_dim, args.n_layers).to(args.device)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

for epoch in range(args.epochs):
    model.train()
    # Simplified BPR Sampling
    perm = torch.randperm(len(u_idx))
    epoch_loss = 0
    
    for i in range(0, len(u_idx), args.batch_size):
        batch_idx = perm[i:i+args.batch_size]
        u = torch.tensor(u_idx[batch_idx]).to(args.device)
        i_pos = torch.tensor(i_idx[batch_idx]).to(args.device)
        i_neg = torch.randint(0, num_items, (len(batch_idx),)).to(args.device)
        
        optimizer.zero_grad()
        loss = model(u, i_pos, i_neg, edge_ui, norm_ui, edge_uu, norm_uu)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{args.epochs}, Loss: {epoch_loss:.4f}")

# Save Model
torch.save({'model_state_dict': model.state_dict(), 'user_map': user_map, 'item_map': item_map}, args.checkpoint_path)
print(f"Model saved to {args.checkpoint_path}")

Loading reviews...
Loading users and social network...
Users: 79345, Items: 9973, Social Edges: 311406
Epoch 1/10, Loss: 17.3475
Epoch 2/10, Loss: 9.8516
Epoch 3/10, Loss: 7.4945
Epoch 4/10, Loss: 6.2337
Epoch 5/10, Loss: 5.4787
Epoch 6/10, Loss: 4.8188
Epoch 7/10, Loss: 4.3663
Epoch 8/10, Loss: 3.9790
Epoch 9/10, Loss: 3.6916
Epoch 10/10, Loss: 3.4059
Model saved to social_lightgcn_model.pt
